# QC — FastQC + MultiQC

Один универсальный QC-ноутбук для всех наборов данных и стадий.

QC не создаёт отдельные каталоги `qc_<stage>`, а складывает результаты **внутрь самой стадии**:

- `raw/<dataset>/qc/{fastqc,multiqc}`
- `results/<dataset>/trimmed/qc/{fastqc,multiqc}`
- `results/<dataset>/pr_trimmed/qc/{fastqc,multiqc}`
- `results/<dataset>/merged/qc/{fastqc,multiqc}`
- `results/<dataset>/post_annotation_filtered/qc/{fastqc,multiqc}`
- `results/<dataset>/simulated/<branch>/qc/{fastqc,multiqc}`

Для обычного запуска изменяются `DATASET` и `STAGE`. Для стадии `simulated` дополнительно указывается `SIMULATION_BRANCH`.


In [ ]:
import os, sys, sysconfig
from pathlib import Path

# На сервере используем bcr_env, если он существует. На локальном Mac
# оставляем активное Python/Jupyter окружение без подмены PATH.
_CONDA_ENV = Path("/data/user/epishkin/conda/envs/bcr_env")
if _CONDA_ENV.is_dir():
    os.environ["PATH"] = str(_CONDA_ENV / "bin") + ":" + os.environ.get("PATH", "")
    os.environ["PYTHONNOUSERSITE"] = "1"
    sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
    for _site in [str(_CONDA_ENV / "lib/python3.11/site-packages"), sysconfig.get_path("purelib")]:
        if os.path.isdir(_site) and _site not in sys.path:
            sys.path.insert(0, _site)
    os.environ["HOME"] = "/data/user/epishkin"
    os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
    os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)


In [ ]:
import shutil
import subprocess
from pathlib import Path

def log(msg):
    print(f"[qc] {msg}")

VALID_STAGES = {
    "raw",
    "trimmed",
    "pr_trimmed",
    "merged",
    "post_annotation_filtered",
    "simulated",
}

def stage_dir(volume, dataset, stage, simulation_branch=None):
    volume = Path(volume)
    if stage not in VALID_STAGES:
        raise ValueError(f"stage must be one of {sorted(VALID_STAGES)}")

    if stage == "raw":
        return volume / "raw" / dataset

    base = volume / "results" / dataset
    if stage == "simulated":
        if not simulation_branch:
            raise ValueError("SIMULATION_BRANCH is required when STAGE='simulated'")
        return base / "simulated" / simulation_branch

    return base / stage

def fastq_dir(volume, dataset, stage, simulation_branch=None):
    root = stage_dir(volume, dataset, stage, simulation_branch)
    if stage == "raw":
        return root
    if stage == "simulated":
        return root / "06_fastq_pe150"
    return root / "fastq"

def qc_dir(volume, dataset, stage, simulation_branch=None):
    return stage_dir(volume, dataset, stage, simulation_branch) / "qc"

def run_qc(
    volume,
    dataset,
    stage,
    threads=4,
    simulation_branch=None,
    force=True,
):
    root = stage_dir(volume, dataset, stage, simulation_branch)
    sd = fastq_dir(volume, dataset, stage, simulation_branch)

    if not root.is_dir():
        raise FileNotFoundError(f"Stage directory not found: {root}")
    if not sd.is_dir():
        raise FileNotFoundError(f"Input FASTQ directory not found: {sd}")

    base = qc_dir(volume, dataset, stage, simulation_branch)
    fastqc_out = base / "fastqc"
    multiqc_out = base / "multiqc"

    # Сохранять все остальные QC-артефакты конкретной стадии. В частности,
    # simulated/<branch>/qc может уже содержать TSV с truth/PCR/allocation.
    if force:
        for d in (fastqc_out, multiqc_out):
            if d.exists():
                shutil.rmtree(d)
    else:
        existing = [str(d) for d in (fastqc_out, multiqc_out) if d.exists()]
        if existing:
            raise FileExistsError(
                "QC output already exists; set force=True to replace only "
                f"FastQC/MultiQC subdirectories: {existing}"
            )

    fastqc_out.mkdir(parents=True, exist_ok=True)
    multiqc_out.mkdir(parents=True, exist_ok=True)

    fastqs = sorted(sd.glob("*.fastq.gz")) + sorted(sd.glob("*.fastq"))
    if not fastqs:
        raise FileNotFoundError(f"No FASTQ files found in {sd}")

    label = stage if stage != "simulated" else f"simulated_{simulation_branch}"
    report_name = f"{dataset}_{label}_multiqc"

    log(f"=== QC {dataset} ({label}): {len(fastqs)} files ===")
    log(f"Stage: {root}")
    log(f"Input: {sd}")
    log(f"QC output: {base}")

    subprocess.run(
        [
            "fastqc",
            "-t", str(threads),
            "-q",
            "--noextract",
            *map(str, fastqs),
            "-o", str(fastqc_out),
        ],
        check=True,
    )
    subprocess.run(
        [
            "multiqc",
            str(fastqc_out),
            "-o", str(multiqc_out),
            "-n", report_name,
            "-f",
        ],
        check=True,
    )

    report = multiqc_out / f"{report_name}.html"
    log(f"Report: {report}")
    return report


### Текущая конфигурация — моделированные данные человека

Для другой обычной стадии достаточно изменить `DATASET` и `STAGE`.


In [ ]:
PROJECT_ROOT = Path("/Users/epishkin/workspace/bcr-assembler")
DATASET = "PRJEB30386"
STAGE = "simulated"

# Используется только при STAGE == "simulated".
SIMULATION_BRANCH = "insilicoseq_150bp_novaseq_post_annotation_filtered"

THREADS = 4

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET)
print("Stage:", STAGE)
print("Simulation branch:", SIMULATION_BRANCH)
print("Stage dir:", stage_dir(PROJECT_ROOT, DATASET, STAGE, SIMULATION_BRANCH))
print("Input:", fastq_dir(PROJECT_ROOT, DATASET, STAGE, SIMULATION_BRANCH))
print("QC output:", qc_dir(PROJECT_ROOT, DATASET, STAGE, SIMULATION_BRANCH))


### Запуск QC

Для текущей конфигурации FastQC/MultiQC будут записаны в:

`results/PRJEB30386/simulated/insilicoseq_150bp_novaseq_post_annotation_filtered/qc/`

Существующие TSV с truth/PCR/allocation в `qc/` не затрагиваются. При `force=True` пересоздаются только `qc/fastqc/` и `qc/multiqc/`.


In [ ]:
report = run_qc(
    PROJECT_ROOT,
    DATASET,
    STAGE,
    threads=THREADS,
    simulation_branch=SIMULATION_BRANCH,
    force=True,
)
report
